# BraTS-PNG three-way comparison — run the AA-CBR harness yourself

Runs the **matched 5-fold CV** AA-CBR sweeps for two feature sources and then builds the
**Baseline vs Slot+AA-CBR vs GT+AA-CBR** comparison (figure + LaTeX table + analysis).

- **GT + AA-CBR** — features from ground-truth segmentation masks (no GPU needed).
- **Slot + AA-CBR** — features from the trained `SlotClassifier2D` checkpoint (uses the GPU if the
  kernel is on a GPU node; works on CPU but slower).
- **Baselines** — read from `baseline/num_classes_leaderboard.json` (already produced by the
  num-classes sweep).

Both sources share the same patients and the same `StratifiedKFold(5, shuffle, seed=0)` split, so
macro-F1 is directly comparable within each `k`. Run the cells top to bottom — re-running is safe,
each sweep cell rewrites its own `leaderboard_*_cv.json` from scratch.

In [1]:
import importlib, json, sys
import torch
from pathlib import Path

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from aacbr import eval_brats_2d_cv as cv
from comparison import compare_pipelines as cmp
importlib.reload(cv); importlib.reload(cmp)

# ── Configure checkpoints to evaluate ────────────────────────────────────────
PATIENT_FRAC = 0.15   # Test set that was held out during training

CHECKPOINTS = [
    {"path": str(FYP_ROOT / "slot_attention/training_2d/models/checkpoints/brats_png_v14a_0.15_test/ckpt.pt"),
     "label": "Fully supervised"},
    {"path": str(FYP_ROOT / "slot_attention/training_2d/models/checkpoints/brats_png_v17_weak_0.15_new_norm/ckpt.pt"),
     "label": "Weakly supervised"},
    {"path": str(FYP_ROOT / "slot_attention/training_2d/models/checkpoints/brats_png_v19a_entropy/ckpt.pt"),
     "label": "Weakly supervised (spatial dice matching)"},
    # add more checkpoints here as needed
]
# ─────────────────────────────────────────────────────────────────────────────

SEED, N_FOLDS, NUM_SLOTS = 0, 5, 5
GT_LB           = FYP_ROOT / 'aacbr' / 'leaderboard_gt_cv.json'
TRAINED_LB      = FYP_ROOT / 'aacbr' / 'leaderboard_trained_cv.json'
SINGLE_SPLIT_LB = FYP_ROOT / 'aacbr' / 'leaderboard_trained.json'  # source of best configs per checkpoint

def best_per_k(sweep):
    for nb in sorted({r['n_bins'] for r in sweep}):
        b = max((r for r in sweep if r['n_bins'] == nb), key=lambda r: r['mean_f1'])
        print(f"  k={nb}: cv_F1={b['mean_f1']:.4f} ± {b['std_f1']:.3f}   "
              f"({b['char']}/{b['agg']}/{b['strategy']}, strict={b['strict']}, smart_default={b['smart_default']})")

_all_pids = sorted(cv.group_by_patient(cv.BraTS2020PNGDataset(data_dir=cv.DATA_DIR, is_train=False)).keys())
_n_total = len(_all_pids)
_n_used = max(1, int(_n_total * PATIENT_FRAC))

print('Setup OK.')
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PATIENT_FRAC={PATIENT_FRAC} | patients={_n_used}/{_n_total} | {len(CHECKPOINTS)} checkpoint(s) configured")
for c in CHECKPOINTS:
    exists = Path(c['path']).exists()
    print(f"  {'[OK]' if exists else '[MISSING]'} {c['label']}: {Path(c['path']).parent.name}")

## 0 · Baseline ResNet — train on 85%, evaluate on held-out 15%

Trains a ResNet-50 end-to-end on the 85% *training* patients (the same patients excluded
from the slot model's training set are **held out**). After each of the 5 CV folds trained
on the 85%, the fold's best checkpoint is evaluated on the same `PATIENT_FRAC` held-out
patients as GT/Slot+AA-CBR, giving `holdout_cv_mean_f1 ± std` for the comparison table.

Results are written to `baseline/num_classes_leaderboard.json`.
**Requires a GPU — run this on an HPC node or skip if the leaderboard is already populated.**

In [2]:
import importlib
from baseline.sweeps import train_end_to_end_2d as bl
from baseline.sweeps.num_classes_aggregate import build_leaderboard
importlib.reload(bl)

BASELINE_DATASET = 'brats_png'
BASELINE_BASE_CFG = dict(
    dataset=BASELINE_DATASET,
    data_dir=bl.DEFAULTS[BASELINE_DATASET]['data_dir'],
    csv_path=bl.DEFAULTS[BASELINE_DATASET]['csv_path'],
    model='resnet50',
    strategy='categorical',
    epochs=50,
    batch_size=4,
    lr=5e-5,
    weight_decay=0.001,
    n_splits=5,
    fold_seed=0,
    seed=0,
    quantile=True,
    in_channels=5,
    holdout_frac=PATIENT_FRAC,
    num_classes_sweep=True,
    max_patients=None,
    allowed_patients=None,
    patience=5,
    scheduler_patience=4,
    min_delta=0.0,
    score_name='QL2',
)

for nb in [2, 3, 4, 5]:
    print(f"\n{'='*60}\nBaseline num_classes sweep: num_bins={nb}\n{'='*60}")
    bl.run_cv({**BASELINE_BASE_CFG, 'num_bins': nb})

build_leaderboard(write=True)
print('Baseline leaderboard updated.')

## 1 · GT + AA-CBR — 5-fold CV on GT's own best configs (fast)

GT features are evaluated on the **best config per (k, strategy) that your GT sweep already found**
(`eval_brats_gt_2d.ipynb` → `aacbr/leaderboard.json`) — 16 configs spanning k=2–5. Those were
selected on a single 80/20 split, so here they're re-run under the **matched 5-fold CV** (same
patients/folds as the trained pipeline), making the macro-F1 directly comparable. No GPU; a few minutes.

This tunes GT *for GT*. To instead score GT on the trained model's configs (a same-config feature
ablation that isolates GT-vs-predicted features), point `GT_CONFIG_SOURCE` at
`aacbr/leaderboard_trained.json` with eval_script `'eval_trained_2d'`.

In [3]:
# GT + AA-CBR on GT's OWN best config per (k, strategy), from your GT sweep
# (eval_brats_gt_2d.ipynb -> aacbr/leaderboard.json). Those were picked on a single
# 80/20 split; here they're re-run under the matched 5-fold CV. 16 configs.
GT_CONFIG_SOURCE = FYP_ROOT / 'aacbr' / 'leaderboard.json'
configs = cv.configs_from_leaderboard(str(GT_CONFIG_SOURCE), 'eval_brats_gt_2d')
print(f'Re-using {len(configs)} GT best-per-(k,strategy) configs from {GT_CONFIG_SOURCE.name}')

GT_LB.unlink(missing_ok=True)  # fresh file -> no duplicate rows on re-run
gt_sweep, gt_cache, gt_kf, gt_cfg = cv.run_configs('gt', configs, N_FOLDS, SEED, NUM_SLOTS,
                                                    patient_frac=PATIENT_FRAC)
cv.log_best_per_bin_strategy('gt', gt_sweep, gt_cache, gt_kf, gt_cfg,
                             N_FOLDS, SEED, NUM_SLOTS, None, str(GT_LB),
                             notes='GT 5-fold CV on GT-own best configs',
                             patient_frac=PATIENT_FRAC)
print('\nGT best-CV-F1 config per k:'); best_per_k(gt_sweep)

## 2 · Slot + AA-CBR — 5-fold CV for each checkpoint

For each checkpoint in `CHECKPOINTS`, loads the best config per (k, strategy) from the existing
single-split sweep in `leaderboard_trained.json` (`eval_trained_2d`), then re-runs those configs
under the same **matched 5-fold CV** as the GT pipeline. Results are written to
`leaderboard_trained_cv.json`.

All checkpoints share the same `StratifiedKFold(5, shuffle, seed=0)` split as GT, so macro-F1
is directly comparable across pipelines within each `k`.

**Prerequisite:** each checkpoint in `CHECKPOINTS` must have a prior sweep in
`leaderboard_trained.json` (produced by `eval_trained_model_brats_2D.ipynb`). The cell
will warn and skip any checkpoint with no matching entries.

In [4]:
TRAINED_LB.unlink(missing_ok=True)  # fresh file -> no duplicate rows on re-run

for ckpt_info in CHECKPOINTS:
    ckpt_path = ckpt_info["path"]
    label = ckpt_info["label"]

    configs = cv.configs_from_leaderboard(str(SINGLE_SPLIT_LB), 'eval_trained_2d',
                                          checkpoint=ckpt_path)
    if not configs:
        print(f"WARNING: no configs found for '{label}' ({Path(ckpt_path).parent.name}) "
              f"-- skipping. Run the single-split sweep in eval_trained_model_brats_2D.ipynb first.")
        continue

    subset_note = f"PATIENT_FRAC={PATIENT_FRAC}" if PATIENT_FRAC < 1.0 else "full dataset"
    print(f"\n-- {label} ({subset_note}) | {len(configs)} configs --")

    sweep, feature_cache, kf, cfg = cv.run_configs(
        'trained', configs, N_FOLDS, SEED, NUM_SLOTS,
        checkpoint=ckpt_path, patient_frac=PATIENT_FRAC)
    cv.log_best_per_bin_strategy(
        'trained', sweep, feature_cache, kf, cfg,
        N_FOLDS, SEED, NUM_SLOTS, ckpt_path, str(TRAINED_LB),
        notes=label, patient_frac=PATIENT_FRAC)
    print(f"\n{label} best-CV-F1 per k:"); best_per_k(sweep)

## 3 · Build the comparison (figure + LaTeX table + analysis)

Reads the two CV leaderboards just written plus the baseline num-classes leaderboard, selects the
best-CV-F1 config per `k` for each AA-CBR pipeline and the best baseline per `k`, and writes
`comparison/figures/`, `comparison/tables/`, `comparison/results/`.

In [5]:
importlib.reload(cmp)
ds = cmp.DATASETS['brats_png']
NAME = 'compare_brats_png'

# Derive checkpoint lists from the CHECKPOINTS config above
SLOT_CHECKPOINTS = [Path(c["path"]).parent.name for c in CHECKPOINTS]
CHECKPOINT_LABELS = {Path(c["path"]).parent.name: c["label"] for c in CHECKPOINTS}

data = cmp.assemble(ds, checkpoint_labels=CHECKPOINT_LABELS, only_checkpoints=SLOT_CHECKPOINTS)
for m in data:
    print(f'{m}: k={sorted(data[m])}')

for sub in ('figures', 'tables', 'results'):
    (cmp.OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
stub = cmp.OUT_DIR / 'figures' / NAME
cmp.make_figure(data, ds['title'], stub)
cmp.make_table(data, 'brats_png', ds['title'], cmp.OUT_DIR / 'tables' / f'{NAME}.tex')
cmp.make_analysis(data, 'brats_png', ds['title'], cmp.OUT_DIR / 'tables' / f'analysis_{NAME}.tex')
(cmp.OUT_DIR / 'results' / f'{NAME}.json').write_text(
    json.dumps({'title': ds['title'], 'num_classes': cmp.NUM_CLASSES, 'data': data}, indent=2))

from IPython.display import Image, display
display(Image(filename=str(stub) + '_f1.png'))

## 4 · F1 comparison table

Rendered macro-F1 table (green = higher, **bold** = best method per *k*) and the selected
AA-CBR configuration per *k*. Also written to `comparison/tables/compare_brats_png.csv`
(long format, with per-fold std) and as the LaTeX `table` in the final cell.

In [6]:
# F1 comparison table (rendered) + CSV export
import pandas as pd
NAME = 'compare_brats_png'
df_f1  = cmp.to_dataframe(data)
df_cfg = cmp.config_dataframe(data)
cmp.write_csv(data, cmp.OUT_DIR / 'tables' / f'{NAME}.csv')

display(
    df_f1.style
      .format('{:.3f}', na_rep='—')
      .background_gradient(cmap='RdYlGn', axis=None)
      .highlight_max(axis=0, subset=pd.IndexSlice[list(data.keys()), :], props='font-weight:bold')
      .set_caption('Macro-F1 by method × number of classes  '
                   '(green = higher; bold = best method per k; per-fold std in the CSV / LaTeX table)')
)
print('Best-CV-F1 AA-CBR configuration per k (characterisation / aggregation / strategy):')
display(df_cfg)

In [7]:
print('===== LaTeX table =====\n')
print((cmp.OUT_DIR / 'tables' / f'{NAME}.tex').read_text())